# Cell 1. Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


# Cell 2. Widget Definitions

In [0]:
# Catalog & Schema setup
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("bronze_schema", "bronze_ingestion")
dbutils.widgets.text("silver_schema", "silver_ingestion")

# Table Names
dbutils.widgets.text("ecb_bronze_table", "brz_macro_ecb_fx_daily")
dbutils.widgets.text("fred_bronze_table", "brz_macro_fred_series")
dbutils.widgets.text("ecb_silver_table", "slv_macro_ecb_fx_daily")
dbutils.widgets.text("fred_silver_table", "slv_macro_fred_series")

# Filters (Optional)
dbutils.widgets.text("ecb_quote_ccy_filter", "USD,GBP,CNY") 
dbutils.widgets.text("fred_series_filter", "DFF,CPIAUCSL,DTWEXBGS")

# Cell 3. Configuration Parsing

In [0]:
CATALOG = dbutils.widgets.get("catalog").strip()
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema").strip()
SILVER_SCHEMA = dbutils.widgets.get("silver_schema").strip()

ECB_BRONZE_TBL  = f"{CATALOG}.{BRONZE_SCHEMA}.{dbutils.widgets.get('ecb_bronze_table').strip()}"
FRED_BRONZE_TBL = f"{CATALOG}.{BRONZE_SCHEMA}.{dbutils.widgets.get('fred_bronze_table').strip()}"
ECB_SILVER_TBL  = f"{CATALOG}.{SILVER_SCHEMA}.{dbutils.widgets.get('ecb_silver_table').strip()}"
FRED_SILVER_TBL = f"{CATALOG}.{SILVER_SCHEMA}.{dbutils.widgets.get('fred_silver_table').strip()}"

# Parse filters into lists
ecb_filter = [s.strip().upper() for s in dbutils.widgets.get("ecb_quote_ccy_filter").split(",") if s.strip()]
fred_filter = [s.strip().upper() for s in dbutils.widgets.get("fred_series_filter").split(",") if s.strip()]

print(f"[INFO] Config Ready:")
print(f"       ECB:  {ECB_BRONZE_TBL} -> {ECB_SILVER_TBL}")
print(f"       FRED: {FRED_BRONZE_TBL} -> {FRED_SILVER_TBL}")

# Cell 4. ECB Pipeline (Logic + Write)

In [0]:
print("[INFO] Processing ECB FX Data...")

if not spark.catalog.tableExists(ECB_BRONZE_TBL):
    print(f"[WARN] Bronze table {ECB_BRONZE_TBL} does not exist. Skipping ECB pipeline.")
else:
    # 1. DDL: Ensure Target Exists
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {ECB_SILVER_TBL} (
      rate_date      DATE,
      base_ccy       STRING,
      quote_ccy      STRING,
      fx_rate        DOUBLE,
      ingestion_ts   TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (rate_date)
    """)

    # 2. Parsing Logic (Regex Strategy with Quote Fix)
    chunks = (spark.table(ECB_BRONZE_TBL)
      .select("base_ccy", "raw_json", "ingestion_ts")
      
      # 将 XML 中的单引号(') 统一替换为双引号(")，为了后续的 split 逻辑
      .withColumn("clean_xml", F.regexp_replace(F.col("raw_json"), "'", '"'))

      # 切割时间块
      .withColumn("block", F.explode(F.split(F.col("clean_xml"), 'time="')))
      
      # 过滤出包含日期的块 (格式: YYYY-MM-DD)
      .where(F.col("block").rlike(r"^\d{4}-\d{2}-\d{2}")) 
      
      # 提取日期
      .withColumn("rate_date", F.to_date(F.regexp_extract(F.col("block"), r"^(\d{4}-\d{2}-\d{2})", 1)))
      
      # 二次切割：提取货币对块
      .withColumn("pair", F.explode(F.split(F.col("block"), '<Cube currency="')))
      .where(F.col("pair").rlike(r'^[A-Z]{3}" rate="'))
      
      # 正则提取：货币代码和汇率
      .withColumn("quote_ccy", F.regexp_extract(F.col("pair"), r'^([A-Z]{3})"', 1))
      .withColumn("fx_rate", F.regexp_extract(F.col("pair"), r'rate="([0-9.]+)"', 1).cast("double"))
      
      .select("rate_date", "base_ccy", "quote_ccy", "fx_rate", "ingestion_ts")
    )

    # 3. Apply Filters
    if ecb_filter:
        chunks = chunks.where(F.col("quote_ccy").isin(ecb_filter))

    # 4. Deduplicate (Keep latest ingestion)
    wkey = Window.partitionBy("rate_date", "base_ccy", "quote_ccy").orderBy(F.col("ingestion_ts").desc())
    ecb_final = chunks.withColumn("rn", F.row_number().over(wkey)).where(F.col("rn") == 1).drop("rn")

    # 5. Merge (Upsert)
    ecb_final.createOrReplaceTempView("stg_ecb_fx")
    print(f"[INFO] Merging ECB rows into Silver table...")

    spark.sql(f"""
    MERGE INTO {ECB_SILVER_TBL} AS tgt
    USING stg_ecb_fx AS src
    ON  tgt.rate_date = src.rate_date AND tgt.base_ccy = src.base_ccy AND tgt.quote_ccy = src.quote_ccy
    WHEN MATCHED THEN UPDATE SET tgt.fx_rate = src.fx_rate, tgt.ingestion_ts = src.ingestion_ts
    WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"[SUCCESS] ECB Merge completed into {ECB_SILVER_TBL}")

# Cell 5. FRED Pipeline (Logic + Write)

In [0]:
print("[INFO] Processing FRED Series Data...")

if not spark.catalog.tableExists(FRED_BRONZE_TBL):
    print(f"[WARN] Bronze table {FRED_BRONZE_TBL} does not exist. Skipping FRED pipeline.")
else:
    # 1. DDL
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FRED_SILVER_TBL} (
      series_id     STRING,
      obs_date      DATE,
      value         DOUBLE,
      ingestion_ts  TIMESTAMP
    )
    USING DELTA
    PARTITIONED BY (series_id) 
    """) 

    # 2. Parsing Logic (JSON Strategy via explode/from_json)
    fred_schema = T.StructType([
        T.StructField("observations", T.ArrayType(
            T.StructType([
                T.StructField("date", T.StringType(), True),
                T.StructField("value", T.StringType(), True)
            ])
        ), True)
    ])

    fred_parsed = (spark.table(FRED_BRONZE_TBL)
      .select("series_id", "raw_json", "ingestion_ts")
      .withColumn("payload", F.from_json(F.col("raw_json"), fred_schema))
      .withColumn("obs", F.explode_outer(F.col("payload.observations")))
      .select(
          F.col("series_id"),
          F.to_date(F.col("obs.date")).alias("obs_date"),
          # Clean out FRED missing dots and invalid formats
          F.when(F.col("obs.value").isin(".", "nan", "NaN", ""), None)
           .otherwise(F.col("obs.value")).cast("double").alias("value"),
          F.col("ingestion_ts")
      )
      .filter(F.col("obs_date").isNotNull())
      .filter(F.col("value").isNotNull()) 
    )

    # 3. Apply Filters
    if fred_filter:
        fred_parsed = fred_parsed.where(F.col("series_id").isin(fred_filter))

    # 4. Deduplicate (Keep latest ingestion for the same date/series)
    wkey_fred = Window.partitionBy("series_id", "obs_date").orderBy(F.col("ingestion_ts").desc())
    fred_final = fred_parsed.withColumn("rn", F.row_number().over(wkey_fred)).where(F.col("rn") == 1).drop("rn")

    # 5. Merge (Upsert)
    fred_final.createOrReplaceTempView("stg_fred_series")
    print(f"[INFO] Merging FRED rows into Silver table...")

    spark.sql(f"""
    MERGE INTO {FRED_SILVER_TBL} AS tgt
    USING stg_fred_series AS src
    ON  tgt.series_id = src.series_id AND tgt.obs_date = src.obs_date
    WHEN MATCHED THEN UPDATE SET tgt.value = src.value, tgt.ingestion_ts = src.ingestion_ts
    WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"[SUCCESS] FRED Merge completed into {FRED_SILVER_TBL}")

# Cell 6. Validation

In [0]:
print("--- 🇪🇺 ECB Summary ---")
if spark.catalog.tableExists(ECB_SILVER_TBL):
    display(spark.sql(f"""
        SELECT rate_date, base_ccy, quote_ccy, fx_rate 
        FROM {ECB_SILVER_TBL} 
        ORDER BY rate_date DESC 
        LIMIT 10
    """))
else:
    print("ECB table not available.")

# COMMAND ----------

print("--- 🇺🇸 FRED Summary ---")
if spark.catalog.tableExists(FRED_SILVER_TBL):
    display(spark.sql(f"""
        SELECT 
            series_id, 
            COUNT(*) as records_count, 
            MAX(obs_date) as latest_date,
            MIN(obs_date) as earliest_date
        FROM {FRED_SILVER_TBL} 
        GROUP BY series_id
        ORDER BY series_id
    """))
else:
    print("FRED table not available.")